# init

> Router initialization for the structure decomposition workflow

In [ ]:
#| default_exp routes.init

In [ ]:
#| export
from fasthtml.common import APIRouter

from cjm_fasthtml_card_stack.core.models import CardStackUrls

from cjm_fasthtml_workflow_transcript_decomp.routes.models import (
    DecompUrls, SelectionUrls, AlignmentUrls,
)

# Import workflow-level handlers
from cjm_fasthtml_workflow_transcript_decomp.routes.core import (
    _handle_current_status,
    _handle_reset,
    _handle_get_sources,
    _handle_switch_chrome,
    _handle_audio_src,
)

# Import Phase 1 (selection) handlers from sub-package
from cjm_fasthtml_workflow_transcript_decomp.routes.selection.queue import (
    _handle_selection_add,
    _handle_selection_remove,
    _handle_selection_reorder,
    _handle_selection_clear,
    _handle_selection_select_all,
    _handle_selection_preview,
)
from cjm_fasthtml_workflow_transcript_decomp.routes.selection.filtering import (
    _handle_source_filter,
    _handle_grouping_change,
    _handle_selection_toggle_focused,
    _handle_keyboard_reorder,
)
from cjm_fasthtml_workflow_transcript_decomp.routes.selection.local_files import (
    _handle_browse_directory,
    _handle_add_external_source,
    _handle_remove_external_source,
)
from cjm_fasthtml_workflow_transcript_decomp.routes.selection.tabs import (
    _handle_tab_switch,
)

# Import Phase 2 (decomposition) handlers — card stack UI operations
from cjm_fasthtml_workflow_transcript_decomp.routes.decomposition.card_stack import (
    _handle_decomp_navigate,
    _handle_decomp_enter_split_mode,
    _handle_decomp_exit_split_mode,
    _handle_decomp_update_viewport,
    _handle_decomp_save_width,
)

# Import Phase 2 (decomposition) handlers — workflow-specific operations
from cjm_fasthtml_workflow_transcript_decomp.routes.decomposition.handlers import (
    _handle_decomp_init,
    _handle_decomp_split,
    _handle_decomp_merge,
    _handle_decomp_undo,
    _handle_decomp_reset,
    _handle_decomp_ai_split,
)

# Import Phase 2 (alignment) handlers — card stack UI operations
from cjm_fasthtml_workflow_transcript_decomp.routes.alignment.card_stack import (
    _handle_align_navigate,
    _handle_align_update_viewport,
    _handle_align_save_width,
)

# Import Phase 2 (alignment) handlers — workflow-specific operations
from cjm_fasthtml_workflow_transcript_decomp.routes.alignment.handlers import (
    _handle_align_init,
    _handle_align_toggle_assign,
    _handle_align_auto_align,
    _handle_align_clear_assignments,
    _handle_align_undo,
)

from cjm_fasthtml_workflow_transcript_decomp.workflow.workflow import StructureDecompWorkflow

## Router Initialization

The `init_router` function creates and configures the APIRouter with all workflow routes.

In [ ]:
#| export
def init_router(
    workflow:StructureDecompWorkflow  # The workflow instance
) -> APIRouter:  # Configured APIRouter with all workflow routes
    """Initialize and return the workflow's API router."""
    router = APIRouter(prefix=workflow.config.route_prefix)

    # -------------------------------------------------------------------------
    # Core Routes
    # -------------------------------------------------------------------------

    @router
    def current_status(request, sess):
        """Get current workflow status."""
        return _handle_current_status(workflow, request, sess)

    @router
    def reset(request, sess):
        """Reset workflow to beginning."""
        return _handle_reset(workflow, request, sess)

    @router
    def get_sources(request, provider_id: str = None, limit: int = 50):
        """Get available transcription sources."""
        return _handle_get_sources(workflow, request, provider_id, limit)

    # -------------------------------------------------------------------------
    # Combined Step Routes (shared chrome switching, audio serving)
    # Defined early so their .to() URLs are available for alignment URL bundle.
    # -------------------------------------------------------------------------

    @router
    async def switch_chrome(request, sess):
        """Switch shared chrome content based on active column."""
        # URL bundles accessed via workflow attributes (set later in this function)
        return await _handle_switch_chrome(
            workflow, request, sess,
            decomp_urls=workflow._decomp_urls,
            align_urls=workflow._align_urls,
        )

    @router
    def audio_src(request, sess):
        """Serve the audio file for alignment playback."""
        return _handle_audio_src(workflow, sess)

    # -------------------------------------------------------------------------
    # Selection Management Routes
    # -------------------------------------------------------------------------

    @router
    def selection_add(request, sess, record_id: str, provider_id: str):
        """Add a source to the selection queue."""
        return _handle_selection_add(
            workflow, request, sess, record_id, provider_id, urls=_selection_urls,
        )

    @router
    def selection_remove(request, sess, record_id: str):
        """Remove a source from the selection queue."""
        return _handle_selection_remove(
            workflow, request, sess, record_id, urls=_selection_urls,
        )

    @router
    async def selection_reorder(request, sess):
        """Reorder items in the selection queue."""
        return await _handle_selection_reorder(
            workflow, request, sess, urls=_selection_urls,
        )

    @router
    def selection_clear(request, sess):
        """Clear all items from the selection queue."""
        return _handle_selection_clear(
            workflow, request, sess, urls=_selection_urls,
        )

    @router
    def selection_select_all(request, sess, group_key: str, grouping_mode: str = "media_path"):
        """Select all transcriptions for a given group."""
        return _handle_selection_select_all(
            workflow, request, sess, group_key, grouping_mode, urls=_selection_urls,
        )

    @router
    def selection_preview(request, record_id: str, provider_id: str):
        """Get preview content for a selected source."""
        return _handle_selection_preview(workflow, request, record_id, provider_id)

    # -------------------------------------------------------------------------
    # Keyboard and Filtering Routes
    # -------------------------------------------------------------------------

    @router
    def selection_toggle_focused(request, sess, record_id: str, provider_id: str):
        """Toggle selection of the focused row (keyboard shortcut)."""
        return _handle_selection_toggle_focused(
            workflow, request, sess, record_id, provider_id, urls=_selection_urls,
        )

    @router
    def keyboard_reorder(request, sess, record_id: str, provider_id: str, direction: str):
        """Move an item up or down in the queue via keyboard (Shift+Up/Down)."""
        return _handle_keyboard_reorder(
            workflow, request, sess, record_id, provider_id, direction, urls=_selection_urls,
        )

    @router
    def source_filter(request, sess, search: str = ""):
        """Filter source list by search term."""
        return _handle_source_filter(
            workflow, request, sess, search, urls=_selection_urls,
        )

    @router
    def grouping_change(request, sess, grouping_mode: str):
        """Change the grouping mode for the source list."""
        return _handle_grouping_change(
            workflow, request, sess, grouping_mode, urls=_selection_urls,
        )

    # -------------------------------------------------------------------------
    # Local Files Browser Routes
    # -------------------------------------------------------------------------

    @router
    def browse_directory(request, sess, path: str):
        """Browse a directory in the local files browser."""
        return _handle_browse_directory(
            workflow, request, sess, path, urls=_selection_urls,
        )

    @router
    def add_external_source(request, sess, path: str):
        """Add an external database source (select_url target from file browser)."""
        return _handle_add_external_source(
            workflow, request, sess, path, urls=_selection_urls,
        )

    @router
    def remove_external_source(request, sess, db_path: str):
        """Remove an external database source."""
        return _handle_remove_external_source(
            workflow, request, sess, db_path, urls=_selection_urls,
        )

    # -------------------------------------------------------------------------
    # Tab Switching Route
    # -------------------------------------------------------------------------

    @router
    def tab_switch(request, sess, direction: str):
        """Switch between source tabs."""
        return _handle_tab_switch(
            workflow, request, sess, direction, urls=_selection_urls,
        )

    # -------------------------------------------------------------------------
    # Phase 1: Cached URL Bundle
    # -------------------------------------------------------------------------

    _selection_urls = SelectionUrls(
        add=selection_add.to(),
        remove=selection_remove.to(),
        reorder=selection_reorder.to(),
        clear=selection_clear.to(),
        select_all=selection_select_all.to(),
        preview=selection_preview.to(),
        toggle_focused=selection_toggle_focused.to(),
        keyboard_reorder=keyboard_reorder.to(),
        filter=source_filter.to(),
        grouping_change=grouping_change.to(),
        browse_directory=browse_directory.to(),
        add_external=add_external_source.to(),
        remove_external=remove_external_source.to(),
        tab_switch=tab_switch.to(),
    )

    # -------------------------------------------------------------------------
    # Phase 2: Decomposition — Route Definitions
    # -------------------------------------------------------------------------

    @router
    async def decomp_init(request, sess):
        """Initialize segments from Phase 1 selected sources."""
        return await _handle_decomp_init(workflow, request, sess, urls=_decomp_urls)

    @router
    async def decomp_split(request, sess, segment_index: int):
        """Split a segment at the specified word position."""
        return await _handle_decomp_split(
            workflow, request, sess, segment_index, urls=_decomp_urls,
        )

    @router
    def decomp_merge(request, sess, segment_index: int):
        """Merge a segment with the previous segment."""
        return _handle_decomp_merge(
            workflow, request, sess, segment_index, urls=_decomp_urls,
        )

    @router
    def decomp_undo(request, sess):
        """Undo the last decomposition operation."""
        return _handle_decomp_undo(workflow, request, sess, urls=_decomp_urls)

    @router
    def decomp_reset(request, sess):
        """Reset segments to the initial NLTK split result."""
        return _handle_decomp_reset(workflow, request, sess, urls=_decomp_urls)

    @router
    async def decomp_ai_split(request, sess):
        """Re-run AI (NLTK) sentence splitting on all current text."""
        return await _handle_decomp_ai_split(workflow, request, sess, urls=_decomp_urls)

    @router
    def decomp_enter_split(request, sess, segment_index: int):
        """Enter split mode for a specific segment."""
        return _handle_decomp_enter_split_mode(
            workflow, request, sess, segment_index, urls=_decomp_urls,
        )

    @router
    def decomp_exit_split(request, sess):
        """Exit split mode."""
        return _handle_decomp_exit_split_mode(workflow, request, sess, urls=_decomp_urls)

    @router
    def decomp_update_viewport(request, sess, visible_count: int):
        """Update viewport with new card count (full outerHTML swap)."""
        return _handle_decomp_update_viewport(
            workflow, sess, visible_count, urls=_decomp_urls,
        )

    @router
    def decomp_save_width(request, sess, card_width: int):
        """Save card stack width to server state."""
        return _handle_decomp_save_width(workflow, sess, card_width)

    # -------------------------------------------------------------------------
    # Phase 2: Decomposition Navigation Routes
    # -------------------------------------------------------------------------

    @router
    def decomp_nav_up(request, sess):
        """Navigate to previous segment."""
        return _handle_decomp_navigate(workflow, sess, direction="up", urls=_decomp_urls)

    @router
    def decomp_nav_down(request, sess):
        """Navigate to next segment."""
        return _handle_decomp_navigate(workflow, sess, direction="down", urls=_decomp_urls)

    @router
    def decomp_nav_first(request, sess):
        """Navigate to first segment."""
        return _handle_decomp_navigate(workflow, sess, direction="first", urls=_decomp_urls)

    @router
    def decomp_nav_last(request, sess):
        """Navigate to last segment."""
        return _handle_decomp_navigate(workflow, sess, direction="last", urls=_decomp_urls)

    @router
    def decomp_nav_page_up(request, sess):
        """Navigate up by page."""
        return _handle_decomp_navigate(workflow, sess, direction="page_up", urls=_decomp_urls)

    @router
    def decomp_nav_page_down(request, sess):
        """Navigate down by page."""
        return _handle_decomp_navigate(workflow, sess, direction="page_down", urls=_decomp_urls)

    # -------------------------------------------------------------------------
    # Phase 2: Decomposition Cached URL Bundle
    # -------------------------------------------------------------------------

    _decomp_urls = DecompUrls(
        card_stack=CardStackUrls(
            nav_up=decomp_nav_up.to(),
            nav_down=decomp_nav_down.to(),
            nav_first=decomp_nav_first.to(),
            nav_last=decomp_nav_last.to(),
            nav_page_up=decomp_nav_page_up.to(),
            nav_page_down=decomp_nav_page_down.to(),
            update_viewport=decomp_update_viewport.to(),
            save_width=decomp_save_width.to(),
        ),
        split=decomp_split.to(),
        merge=decomp_merge.to(),
        enter_split=decomp_enter_split.to(),
        exit_split=decomp_exit_split.to(),
        reset=decomp_reset.to(),
        ai_split=decomp_ai_split.to(),
        undo=decomp_undo.to(),
        init=decomp_init.to(),
    )

    # -------------------------------------------------------------------------
    # Phase 2: Alignment — Route Definitions
    # -------------------------------------------------------------------------

    @router
    async def align_init(request, sess):
        """Initialize alignment from audio file via VAD plugin."""
        return await _handle_align_init(workflow, request, sess, urls=_align_urls)

    @router
    def align_toggle_assign(request, sess, chunk_index: int):
        """Toggle assignment of a VAD chunk to the focused text segment."""
        return _handle_align_toggle_assign(
            workflow, request, sess, chunk_index, urls=_align_urls,
        )

    @router
    def align_auto_align(request, sess):
        """Auto-align all VAD chunks to segments."""
        return _handle_align_auto_align(workflow, request, sess, urls=_align_urls)

    @router
    def align_clear_assignments(request, sess):
        """Clear all VAD chunk assignments."""
        return _handle_align_clear_assignments(
            workflow, request, sess, urls=_align_urls,
        )

    @router
    def align_undo(request, sess):
        """Undo the last alignment operation."""
        return _handle_align_undo(workflow, request, sess, urls=_align_urls)

    # -------------------------------------------------------------------------
    # Phase 2: Alignment Navigation Routes
    # -------------------------------------------------------------------------

    @router
    def align_nav_up(request, sess):
        """Navigate to previous VAD chunk."""
        return _handle_align_navigate(workflow, sess, direction="up", urls=_align_urls)

    @router
    def align_nav_down(request, sess):
        """Navigate to next VAD chunk."""
        return _handle_align_navigate(workflow, sess, direction="down", urls=_align_urls)

    @router
    def align_nav_first(request, sess):
        """Navigate to first VAD chunk."""
        return _handle_align_navigate(workflow, sess, direction="first", urls=_align_urls)

    @router
    def align_nav_last(request, sess):
        """Navigate to last VAD chunk."""
        return _handle_align_navigate(workflow, sess, direction="last", urls=_align_urls)

    @router
    def align_nav_page_up(request, sess):
        """Navigate up by page in alignment."""
        return _handle_align_navigate(workflow, sess, direction="page_up", urls=_align_urls)

    @router
    def align_nav_page_down(request, sess):
        """Navigate down by page in alignment."""
        return _handle_align_navigate(workflow, sess, direction="page_down", urls=_align_urls)

    @router
    def align_update_viewport(request, sess, visible_count: int):
        """Update alignment viewport with new card count."""
        return _handle_align_update_viewport(
            workflow, sess, visible_count, urls=_align_urls,
        )

    @router
    def align_save_width(request, sess, card_width: int):
        """Save alignment card stack width to server state."""
        return _handle_align_save_width(workflow, sess, card_width)

    # -------------------------------------------------------------------------
    # Phase 2: Alignment Cached URL Bundle
    # -------------------------------------------------------------------------

    _align_urls = AlignmentUrls(
        card_stack=CardStackUrls(
            nav_up=align_nav_up.to(),
            nav_down=align_nav_down.to(),
            nav_first=align_nav_first.to(),
            nav_last=align_nav_last.to(),
            nav_page_up=align_nav_page_up.to(),
            nav_page_down=align_nav_page_down.to(),
            update_viewport=align_update_viewport.to(),
            save_width=align_save_width.to(),
        ),
        toggle_assign=align_toggle_assign.to(),
        auto_align=align_auto_align.to(),
        clear_assignments=align_clear_assignments.to(),
        undo=align_undo.to(),
        init=align_init.to(),
        audio_src=audio_src.to(),
    )

    # -------------------------------------------------------------------------
    # Store Route References and URL Bundles
    # -------------------------------------------------------------------------

    workflow._selection_urls = _selection_urls
    workflow._decomp_urls = _decomp_urls
    workflow._align_urls = _align_urls
    workflow._switch_chrome_url = switch_chrome.to()

    workflow._selection_routes = {
        "add": selection_add,
        "remove": selection_remove,
        "reorder": selection_reorder,
        "clear": selection_clear,
        "select_all": selection_select_all,
        "preview": selection_preview,
        "toggle_focused": selection_toggle_focused,
        "keyboard_reorder": keyboard_reorder,
        "filter": source_filter,
        "grouping_change": grouping_change,
        "browse_directory": browse_directory,
        "add_external": add_external_source,
        "remove_external": remove_external_source,
        "tab_switch": tab_switch,
    }

    workflow._decomposition_routes = {
        "init": decomp_init,
        "split": decomp_split,
        "merge": decomp_merge,
        "undo": decomp_undo,
        "reset": decomp_reset,
        "ai_split": decomp_ai_split,
        "enter_split": decomp_enter_split,
        "exit_split": decomp_exit_split,
        "update_viewport": decomp_update_viewport,
        "save_width": decomp_save_width,
        "nav_up": decomp_nav_up,
        "nav_down": decomp_nav_down,
        "nav_first": decomp_nav_first,
        "nav_last": decomp_nav_last,
        "nav_page_up": decomp_nav_page_up,
        "nav_page_down": decomp_nav_page_down,
    }

    workflow._alignment_routes = {
        "init": align_init,
        "toggle_assign": align_toggle_assign,
        "auto_align": align_auto_align,
        "clear_assignments": align_clear_assignments,
        "undo": align_undo,
        "nav_up": align_nav_up,
        "nav_down": align_nav_down,
        "nav_first": align_nav_first,
        "nav_last": align_nav_last,
        "nav_page_up": align_nav_page_up,
        "nav_page_down": align_nav_page_down,
        "update_viewport": align_update_viewport,
        "save_width": align_save_width,
    }

    return router

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()